In [ ]:
import os
from dotenv import load_dotenv
import psycopg2

In [ ]:
load_dotenv()

In [ ]:
os.environ.get("GMAIL_PASSWORD")

In [ ]:
passw=os.environ.get("supabase_pass","")
passw

In [ ]:
conn = psycopg2.connect("postgresql://postgres:{}@db.engepyysrjkmhxkumyit.supabase.co:5432/postgres".format(passw))
cursor = conn.cursor()

In [ ]:
cursor.execute("SELECT * FROM users_database;")
rows = cursor.fetchall()
print(rows)

In [ ]:
rows[1]

In [ ]:
test=[{'category': 'Sports', 'subcategories': ['Football', 'Hockey']},
  {'category': 'Tech', 'subcategories': ['AI', 'Cybersecurity']},
  {'category': 'Finance',
   'subcategories': ['Crypto & Blockchain', 'Central Banks & Policy']},
  {'category': 'Science', 'subcategories': ['General Science News']},
  {'category': 'Policy', 'subcategories': ['Indian Goverment']}]

In [ ]:
multi_preference_category=[{"category": "Tech", "preference": "AI"},{"category":"Sports","preference":"Football"},{"category": "Science", "preference": "Biology & Medicine"}]
multi_preference_category

In [ ]:
new_set=[]

In [ ]:
for item in test:
    no_items = len(item['subcategories'])
    for i in range(no_items):
        rs=dict({"category":item["category"],"preference":item["subcategories"][i]})
        new_set.append(rs)

In [ ]:
new_set

In [ ]:
import json

In [ ]:
ch = json.dumps([
    {"category": "Sports", "preferences": "Football"},
    {"category": "Sports", "preferences": "Cricket"}
])


In [ ]:
ch

In [ ]:
conn.rollback()  # clears the broken transaction

In [ ]:
cursor.execute('INSERT INTO "user" (name, email, categories) VALUES (%s, %s, %s)', ("Alice", "alice@example.com", ch))
conn.commit()

In [ ]:
cursor.execute('SELECT * FROM "user"')
rows = cursor.fetchall()

for row in rows:
    print(row)

In [ ]:
cursor.execute('SELECT preferences FROM "users_database" WHERE email = %s', ("soumya.ranjan.bhoi0011@gmail.com",))
row = cursor.fetchone()
print(row)

In [ ]:
type(row)

In [ ]:
_,_,k=row

In [ ]:
%pwd

In [ ]:
os.chdir("../")

In [ ]:
def modify_data(user_data):
    new_set=[]
    for item in user_data:
        no_items = len(item['subcategories'])
        for i in range(no_items):
            rs=dict({"category":item["category"],"preference":item["subcategories"][i]})
            new_set.append(rs)
    return new_set

In [ ]:
_,_,_,pref=rows[1]

In [ ]:
pref

In [ ]:
modify_data(pref)

In [ ]:
conn.rollback() 

In [ ]:
from src.app.all_function_2 import Workflow2

In [ ]:
obj =Workflow2()

In [ ]:
workflow = obj.build_final_workflow()

In [ ]:
def gen_summary(email):
    cursor.execute('SELECT * FROM "users_database" WHERE email = %s', (email,))
    row = cursor.fetchone()
    _,_,_,pref=row
    new_data=modify_data(pref)
    print(new_data)
    res=workflow.invoke({"items":new_data})
    return res

In [ ]:
items = [{'category': 'Sports', 'preference': 'Football'}, {'category': 'Sports', 'preference': 'Hockey'}, {'category': 'Tech', 'preference': 'AI'}, {'category': 'Tech', 'preference': 'Cybersecurity'}, {'category': 'Finance', 'preference': 'Crypto & Blockchain'}, {'category': 'Finance', 'preference': 'Central Banks & Policy'}, {'category': 'Science', 'preference': 'General Science News'}, {'category': 'Policy', 'preference': 'Indian Goverment'}]

In [ ]:
ans=workflow.invoke({"items":items})

In [ ]:
ans['final_res']['Sports_Football']['filtered_cnt'].items

In [ ]:
ans['final_res']['Sports_Football']['final_output'].details

In [ ]:
ans['final_res']['Sports_Hockey']['final_output'].details

In [ ]:
ans['final_res']['Sports_Hockey']['filtered_cnt'].items

In [ ]:
ans['final_res']['Tech_Cybersecurity']['final_output'].details

In [ ]:
trail=ans['final_res']['Policy_Indian Goverment']['final_output'].details
trail

In [ ]:
def convert_data(items):
    return [
    {"title":item.title , "url": item.url ,"source":item.source, "category":item.category,"sub_category":item.preference,"summary":item.summary,"is_breaking":item.is_breaking,"score":item.score}
    for item in items ]

In [ ]:
all_res=[]

In [ ]:
for all in ans['final_res']:
    final_re=convert_data(ans['final_res'][all]['final_output'].details)
    all_res.append(final_re[0])


In [ ]:
all_res

In [ ]:
CATEGORY_ICONS = {
    "Sports":  "⚽",
    "Tech":    "💻",
    "Finance": "📈",
    "Science": "🔬",
    "Policy":  "🏛️",
}
 
SUBCATEGORY_ICONS = {
    "Football": "⚽", "Cricket": "🏏", "Basketball": "🏀", "Hockey": "🏒",
    "AI": "🤖", "Startups & Business": "🚀", "Mobile & Apps": "📱",
    "Cybersecurity": "🔒", "Stocks & Investing": "📊",
    "Crypto & Blockchain": "🪙", "India-Focused": "🇮🇳",
    "General Finance & Markets": "💰", "Central Banks & Policy": "🏦",
    "Biology & Medicine": "🧬", "Space & Astronomy": "🌌",
    "Physics & Chemistry": "⚗️", "General Science News": "🔭",
    "Indian Government": "🇮🇳", "USA": "🇺🇸",
    "International Policy Bodies": "🌍",
}

In [ ]:
import smtplib
import os
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime
from jinja2 import Environment, FileSystemLoader
from collections import defaultdict

In [ ]:
def build_newsletter_data(user: dict, articles: list) -> dict:

    prefs = user.get("preferences", {})
    filtered = [
        a for a in articles
        if a.get("category") in prefs
        and a.get("sub_category") in prefs.get(a.get("category", ""), [])
    ]

    # Group by category
    grouped = defaultdict(list)
    for article in filtered:
        grouped[article["category"]].append(article)

    # Build category blocks (only categories the user subscribed to)
    categories = []
    for cat_name, art_list in grouped.items():
        # Sort: breaking first, then by score
        art_list.sort(key=lambda x: (not x.get("is_breaking", False),
                                     -x.get("score", 0)))
        categories.append({
            "id":       cat_name.lower().replace(" ", "-"),
            "name":     cat_name,
            "icon":     CATEGORY_ICONS.get(cat_name, "📰"),
            "articles": art_list[:5],   # max 5 per category
        })

    # Determine time of day greeting
    hour = datetime.now().hour
    if hour < 12:
        time_of_day = "morning"
    elif hour < 17:
        time_of_day = "afternoon"
    else:
        time_of_day = "evening"

    return {
        "user_name":      user["name"],
        "user_email":     user["email"],
        "date":           datetime.now().strftime("%A, %d %B %Y"),
        "edition":        datetime.now().strftime("%Y%m%d"),
        "time_of_day":    time_of_day,
        "categories":     categories,
        "total_articles": len(filtered),
        "total_sources":  len(set(a["source"] for a in filtered)),
        "total_topics":   sum(len(v) for v in prefs.values()),
        "breaking_count": sum(1 for a in filtered if a.get("is_breaking")),
    }

In [ ]:
def render_newsletter(data: dict, template_path: str = "newsletter_template.html") -> str:
    """Renders the Jinja2 HTML template with the newsletter data."""
    template_dir  = os.path.dirname(os.path.abspath(template_path))
    template_file = os.path.basename(template_path)

    env      = Environment(loader=FileSystemLoader(template_dir))
    template = env.get_template(template_file)
    return template.render(**data)

In [ ]:
def send_via_gmail(to_email: str, to_name: str,
                   html_content: str, date_str: str):
    GMAIL_USER     = os.getenv("GMAIL_USER",     "")
    GMAIL_PASSWORD = os.getenv("GMAIL_PASSWORD", "")

    msg = MIMEMultipart("alternative")
    msg["Subject"] = f"📡 NewsFlow Digest — {date_str}"
    msg["From"]    = f"NewsFlow <{GMAIL_USER}>"
    msg["To"]      = f"{to_name} <{to_email}>"

    # Attach HTML part
    msg.attach(MIMEText(html_content, "html", "utf-8"))

    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(GMAIL_USER, GMAIL_PASSWORD)
            server.sendmail(GMAIL_USER, to_email, msg.as_string())
        print(f"  ✅ Email sent to {to_email}")
        return True
    except smtplib.SMTPAuthenticationError:
        print("  ❌ Gmail auth failed — check GMAIL_USER and GMAIL_PASSWORD")
        return False
    except Exception as e:
        print(f"  ❌ Failed to send to {to_email}: {e}")
        return False


In [ ]:
def generate_and_send(user: dict, articles: list,
                      template_path: str = "newsletter_template.html"):
    """
    Full pipeline: build data → render HTML → send email.

    Args:
        user:          User profile dict with name, email, preferences
        articles:      List of article dicts from your database
        send_method:   'gmail' or 'sendgrid'
        preview:       If True, also saves HTML file locally
        template_path: Path to the Jinja2 HTML template
    """
    print(f"\n📡 Generating newsletter for {user['name']}...")

    data = build_newsletter_data(user, articles)
    print(f"  Topics: {data['total_topics']} | "
          f"Articles: {data['total_articles']} | "
          f"Breaking: {data['breaking_count']}")

    if data["total_articles"] == 0:
        print("  ⚠️  No articles match user preferences — skipping.")
        return False


    html = render_newsletter(data, template_path)
    print(f"  Rendered {len(html):,} chars of HTML")

  
    return send_via_gmail(user["email"], user["name"],
                              html, data["date"])


In [ ]:
def send_to_all_users(users: list, articles: list):
    print(f"\n📬 Sending newsletters to {len(users)} users...")
    results = {"sent": 0, "failed": 0, "skipped": 0}

    for user in users:
        success = generate_and_send(
            user, articles
        )
        if success is True:
            results["sent"] += 1
        elif success is False:
            results["failed"] += 1
        else:
            results["skipped"] += 1

    print(f"\n✅ Done — Sent: {results['sent']} | "
          f"Failed: {results['failed']} | "
          f"Skipped: {results['skipped']}")
    return results

In [ ]:
test_article=all_res[0]

In [ ]:
test_article

In [ ]:
_,name,email,detils=rows[1]

In [ ]:
email,name

In [ ]:
detils

In [ ]:
preferences={}

In [ ]:
for item in detils:
    preferences[item['category']]=item['subcategories']

In [ ]:
preferences

In [ ]:
test_user={
    "name":name,
    "email":"soumyaranjanbhoi1100@gmail.com",
    "preferences":preferences
}

In [ ]:
generate_and_send(
        user          = test_user,
        articles      = all_res,      
        template_path = "src/Gmail/newsletter_template.html"
    )